In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

print('PyTorch version:', torch.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    device = torch.device('cuda')
else:
    print('No GPU found, using CPU.')
    device = torch.device('cpu')

PyTorch version: 2.14.0+cu130
GPU: NVIDIA GeForce GTX 1650


### KMIST dataset

In [2]:
transform = transforms.Compose([transforms.ToTensor()])

trainset = torchvision.datasets.KMNIST(
    root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.KMNIST(
    root='./data', train=False, download=True, transform=transform)

print(trainset)
print(testset)

Dataset KMNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
           )
Dataset KMNIST
    Number of datapoints: 10000
    Root location: ./data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
           )


In [3]:
BATCH_SIZE = 100 # 60000 datapoints so 600 batches to train
train_loader = torch.utils.data.DataLoader(dataset=trainset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=testset, batch_size=BATCH_SIZE, shuffle=False)

##### Using VAE to get Latent Space Representation

In [20]:
class VAE(nn.Module):
    def __init__(self, x, hl1, hl2, z):  # x->input, hl1 & hl2 hidden layers, z->latent space
        super().__init__()

        # Encoder
        self.fc1 = nn.Linear(x, hl1)
        self.fc2 = nn.Linear(hl1, hl2)
        self.fc_mu = nn.Linear(hl2, z)
        self.fc_log_var = nn.Linear(hl2, z)
        # Decoder
        self.fc3 = nn.Linear(z, hl2)
        self.fc4 = nn.Linear(hl2, hl1)
        self.fc5 = nn.Linear(hl1, x)

    def encoder(self, x_t):
        y = F.relu(self.fc1(x_t))
        y = F.relu(self.fc2(y))
        return self.fc_mu(y), self.fc_log_var(y)  # mu, log_var

    def sampling(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return eps.mul(std).add_(mu)

    def decoder(self, z_t):
        y = F.relu(self.fc3(z_t))
        y = F.relu(self.fc4(y))
        return torch.sigmoid(self.fc5(y))

    def forward(self, x_t):
        mu, log_var = self.encoder(x_t.view(-1, 784))
        z_t = self.sampling(mu, log_var)
        return self.decoder(z_t), mu, log_var

vae = VAE(x=784, hl1=512, hl2=256, z=2).to(device)
if torch.cuda.is_available():
    vae.cuda()
    
print(vae)

VAE(
  (fc1): Linear(in_features=784, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (fc_mu): Linear(in_features=256, out_features=2, bias=True)
  (fc_log_var): Linear(in_features=256, out_features=2, bias=True)
  (fc3): Linear(in_features=2, out_features=256, bias=True)
  (fc4): Linear(in_features=256, out_features=512, bias=True)
  (fc5): Linear(in_features=512, out_features=784, bias=True)
)


In [21]:
def loss_function(recon_x, x, mu, log_var):
    BCE = F.binary_cross_entropy(recon_x, x.view(-1, 784), reduction='sum')
    KLD = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return BCE + KLD

In [26]:
optimizer  = torch.optim.AdamW(vae.parameters())
EPOCHS = 50

train_loss = []
train_accs = []
for epoch in range(EPOCHS):
    vae.train()
    running_loss, running_correct, n_train = 0.0, 0, 0
    for images, _ in train_loader:
        images = images.to(device)
        optimizer.zero_grad()

        recon, mu, log_var = vae(images)
        loss = loss_function(recon, images, mu, log_var)

        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        running_loss += loss.item()
        n_train      += batch_size
        pred   = recon > 0.5
        target = images.view(-1, 784) > 0.5
        running_correct += (pred == target).sum().item()

    train_loss.append(running_loss / n_train)
    train_accs.append(running_correct / (n_train * 784))
    print(f"Epoch {epoch+1}: loss {train_loss[-1]:.4f} | pixel acc {train_accs[-1]:.4f}")

Epoch 1: loss 258.8405 | pixel acc 0.8467
Epoch 2: loss 258.0985 | pixel acc 0.8474
Epoch 3: loss 257.5988 | pixel acc 0.8479
Epoch 4: loss 256.7622 | pixel acc 0.8486
Epoch 5: loss 256.0964 | pixel acc 0.8493
Epoch 6: loss 255.5653 | pixel acc 0.8498
Epoch 7: loss 254.9353 | pixel acc 0.8503
Epoch 8: loss 254.3953 | pixel acc 0.8508
Epoch 9: loss 253.7635 | pixel acc 0.8514
Epoch 10: loss 253.5071 | pixel acc 0.8517
Epoch 11: loss 253.0101 | pixel acc 0.8522
Epoch 12: loss 252.7775 | pixel acc 0.8524
Epoch 13: loss 252.1122 | pixel acc 0.8530
Epoch 14: loss 252.0124 | pixel acc 0.8531
Epoch 15: loss 251.6093 | pixel acc 0.8536
Epoch 16: loss 251.2074 | pixel acc 0.8538
Epoch 17: loss 250.9149 | pixel acc 0.8540
Epoch 18: loss 250.5388 | pixel acc 0.8544
Epoch 19: loss 250.1138 | pixel acc 0.8548
Epoch 20: loss 249.9233 | pixel acc 0.8550
Epoch 21: loss 249.6115 | pixel acc 0.8552
Epoch 22: loss 249.4927 | pixel acc 0.8555
Epoch 23: loss 249.2015 | pixel acc 0.8557
Epoch 24: loss 248.9

In [27]:
for epoch in range(EPOCHS):
    vae.eval()
    val_loss = 0
    with torch.no_grad():
        for image, _ in test_loader:
            image = image.cuda()
            recon, mu, log_var = vae(image)

            val_loss += loss_function(recon,image,mu,log_var).item()
    val_loss = val_loss/len(test_loader)
    

In [33]:
def extract(dataset):
    with torch.no_grad():
        vae.eval()
        loader = torch.utils.data.DataLoader(dataset, batch_size=512, shuffle=False)

        latent_space = []
        for x, _ in loader:
            x = x.to(device)
            x = x.view(-1,784)
            mu, log_var = vae.encoder(x) 
            latent_space.append(mu.cpu())
        return torch.cat(latent_space)

Z_train = extract(trainset)
Z_test = extract(testset)
np.save("Z_train.npy", Z_train.numpy())
np.save("Z_test.npy",  Z_test.numpy())
print("Files Saved")


Files Saved
